In [6]:
# ==============================================================================
# Importar as bibliotecas necessárias para manipulação,
# análise e visualização inicial dos dados.
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import ipykernel as ipk

In [9]:
#Leitura dos arquivos CSV do projeto
df_numbers = pd.read_csv("../data/raw/thepudding_allNumbers.csv")

In [10]:
# Descobrindo o tamanho do dataset
display(df_numbers.shape)

(3117, 9)

In [11]:
# Informações gerais sobre o DataFrame, incluindo o número de entradas, tipos de dados e contagem de valores não nulos
df_numbers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3117 entries, 0 to 3116
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   brand        3117 non-null   str    
 1   product      3117 non-null   str    
 2   name         2016 non-null   str    
 3   specific     3117 non-null   str    
 4   lightness    3117 non-null   float64
 5   hex          3117 non-null   str    
 6   lightToDark  2991 non-null   object 
 7   numbers      3117 non-null   float64
 8   id           2991 non-null   float64
dtypes: float64(3), object(1), str(5)
memory usage: 219.3+ KB


In [12]:
#TRADUÇÃO DAS COLUNAS - Para melhor compreensão, as colunas do DataFrame foram traduzidas para o português. A tradução foi feita de forma literal, mantendo o significado original das colunas.
df_numbers.rename(columns={
    'brand': 'marca',
    'product': 'produto',
    'name': 'nome_tom',
    'specific': 'tom_especifico',
    'lightness': 'luminosidade',
    'hex': 'codigo_hex',
    'lightToDark': 'ordem_luminosidade',
    'numbers': 'numero_tom',
    'id': 'id_tom_marca'
}, inplace=True)

In [13]:
df_numbers.head(1000)

,marca,produto,nome_tom,tom_especifico,luminosidade,codigo_hex,ordem_luminosidade,numero_tom,id_tom_marca
0,Makeup Revolution,Conceal & Define Full Coverage Foundation,NaN,F0,0.949020,#F2F2F2,True,0.0,1.0
1,HOURGLASS,Veil Fluid Makeup,Porcelain,No. 0,0.817647,#F6D3AB,True,0.0,2.0
2,TOM FORD,Traceless Soft Matte Foundation,Pearl,0.0,0.850980,#F0D8C2,True,0.0,3.0
3,Armani Beauty,Neo Nude Foundation,NaN,0,0.911765,#F0E8E1,True,0.0,4.0
4,TOM FORD,Traceless Foundation Stick,Pearl,0.0,0.911765,#FDE5D4,True,0.0,5.0
...,...,...,...,...,...,...,...,...,...
995,bareMinerals,BAREPRO Longwear Powder Foundation,Golden Nude,13,0.790196,#F3CDA0,True,13.0,17.0
996,PAT McGRATH LABS,Sublime Perfection Foundation,Light Medium,13,0.796078,#F2C5A4,True,13.0,37.0
997,bareMinerals,BarePRO™ 24 hour Longwear Liquid Foundation wi...,Golden Nude,13,0.749020,#E3C69B,True,13.0,18.0
998,bareMinerals,Matte Loose Powder Mineral Foundation SPF 15,Golden Beige,13,0.658824,#CCA184,True,13.0,38.0


In [14]:
# OBJETIVO: Fazer um raio-X (diagnóstico inicial) de todas as colunas da base de dados.
# Percorre cada coluna do DataFrame 'df_numbers', uma por uma, para analisar sua qualidade e estrutura
for coluna in df_numbers.columns:
    # Imprime o nome da coluna atual formatado entre traços para organização visual no terminal
    print(f"\n--- {coluna} ---")
    # Exibe o tipo de dado armazenado na coluna (ex: str, float64, int64)
    print(f"Tipo: {df_numbers[coluna].dtype}")
    # Conta e exibe a quantidade de valores ÚNICOS (distintos/diferentes) presentes na coluna
    print(f"Valores únicos: {df_numbers[coluna].nunique()}")
    # Conta e exibe a quantidade total de valores NULOS/faltantes (NaN) na coluna
    print(f"Nulos: {df_numbers[coluna].isna().sum()}")


--- marca ---
Tipo: str
Valores únicos: 64
Nulos: 0

--- produto ---
Tipo: str
Valores únicos: 145
Nulos: 0

--- nome_tom ---
Tipo: str
Valores únicos: 681
Nulos: 1101

--- tom_especifico ---
Tipo: str
Valores únicos: 1057
Nulos: 0

--- luminosidade ---
Tipo: float64
Valores únicos: 388
Nulos: 0

--- codigo_hex ---
Tipo: str
Valores únicos: 2940
Nulos: 0

--- ordem_luminosidade ---
Tipo: object
Valores únicos: 2
Nulos: 126

--- numero_tom ---
Tipo: float64
Valores únicos: 425
Nulos: 0

--- id_tom_marca ---
Tipo: float64
Valores únicos: 130
Nulos: 126


Nota-se que neste csv Numbers temos 0 nulos em tom_especifico, diferente do csv de shades, que foi preciso criar uma coluna unificada pra termos uma coluna sem NaN. Porem pra mantermos um padrão nas tabelas, optei por ter tbm uma coluna unificada usando o tom especifico como prioridade

In [15]:
df_numbers[['tom_especifico', 'nome_tom']].head(160)

,tom_especifico,nome_tom
0,F0,NaN
1,No. 0,Porcelain
2,0.0,Pearl
3,0,NaN
4,0.0,Pearl
...,...,...
155,DpW2,NaN
156,02,Fair Ivory
157,02,NaN
158,No. 2,Light Beige


In [16]:
# 1. Cria a cópia para tratamento
df_numbers_limpo = df_numbers.copy()

# 2. Cria a coluna 'tom_unificado' (mantendo a mesma lógica do shades)
# Como no 'numbers' a coluna 'tom_especifico' não tem nulos, ela vai direto
df_numbers_limpo['tom_unificado'] = df_numbers_limpo['tom_especifico']

# As colunas originais 'nome_tom' e 'tom_especifico' permanecem intactas,
# preservando os nulos nativos exatamente como você foi feito na planilha 'shades'.


## Validação da variável `id_tom`

### Objetivo

Avaliar se a variável `id_tom` pode ser utilizada como identificador
único dos registros da base `allNumbers` ou como chave de relacionamento
com a base `allShades`.

A análise considera:

- quantidade de registros;
- quantidade de valores únicos;
- valores nulos;
- quantidade de duplicidades;
- distribuição dos registros por `id_tom`;
- relação entre `id_tom` e as demais variáveis de identificação.

### Resultado preliminar

A base possui 3.117 registros e apenas 130 valores distintos de `id_tom`,
sendo 126 registros nulos.

Entre os 2.991 registros preenchidos, foram identificadas 2.861
ocorrências duplicadas de `id_tom`.

Dessa forma, a variável não apresenta unicidade suficiente para atuar
como identificador individual dos registros ou como chave primária da
tabela.

A investigação de seu significado será realizada antes da decisão
definitiva sobre sua remoção da camada analítica.

In [19]:
# Verificação do comportamento do id_tom
print("Total de registros:", len(df_numbers_limpo))

print("Valores únicos de id_tom:", df_numbers_limpo["id_tom_marca"].nunique())

print("Valores nulos:", df_numbers_limpo["id_tom_marca"].isna().sum())

print(
    "Duplicidades considerando id_tom:",
    df_numbers_limpo["id_tom_marca"].duplicated().sum()
)

Total de registros: 3117
Valores únicos de id_tom: 130
Valores nulos: 126
Duplicidades considerando id_tom: 2986


id_tom não é um identificador único de cada linha da tabela. Portanto, não podemos usar id_tom como chave primária da allNumbers
3.117 registros > 130 IDs diferentes > muitos registros compartilham o mesmo ID

In [20]:
# Registros com id_tom preenchido
df_id = df_numbers_limpo[df_numbers_limpo["id_tom_marca"].notna()]

print("Registros com id_tom:", len(df_id))
print("id_tom únicos:", df_id["id_tom_marca"].nunique())

print(
    "Duplicados entre os id_tom preenchidos:",
    df_id["id_tom_marca"].duplicated().sum()
)

Registros com id_tom: 2991
id_tom únicos: 130
Duplicados entre os id_tom preenchidos: 2861


Dos 2.991 registros que possuem id_tom, somente 130 IDs são diferentes.
2991 REGISTROS > 130 IDs diferentes
Isso significa que 2.861 registros possuem um id_tom que já apareceu anteriormente.

In [23]:
# Ver quais id_tom aparecem em mais de um registro
duplicados_id = (
    df_id.groupby("id_tom_marca")
         .size()
         .sort_values(ascending=False)
)

print(duplicados_id[duplicados_id > 1].head(20))

id_tom_marca
66.0     60
122.0    50
88.0     50
97.0     50
99.0     50
91.0     50
96.0     50
69.0     50
98.0     48
78.0     43
115.0    43
1.0      42
93.0     42
65.0     42
103.0    42
95.0     42
123.0    40
3.0      40
57.0     40
116.0    40
dtype: int64


In [24]:
# ==============================================================================
# INVESTIGAÇÃO DO SIGNIFICADO DO id_tom
# ==============================================================================

# Objetivo:
# Verificar quantos valores diferentes de marca, produto, tom específico
# e número do tom estão associados a cada id_tom.
#
# Isso nos ajuda a descobrir se o id_tom representa:
# - um registro individual;
# - um produto;
# - um grupo de tons;
# - ou alguma outra entidade da base.
# ==============================================================================

analise_id_tom = (
    df_id
    .groupby("id_tom_marca")
    .agg(
        qtd_registros=("id_tom_marca", "size"),
        marcas_diferentes=("marca", "nunique"),
        produtos_diferentes=("produto", "nunique"),
        tons_diferentes=("tom_especifico", "nunique"),
        numeros_tom_diferentes=("numero_tom", "nunique")
    )
    .sort_values("qtd_registros", ascending=False)
)

analise_id_tom.head(20)

,qtd_registros,marcas_diferentes,produtos_diferentes,tons_diferentes,numeros_tom_diferentes
id_tom_marca,,,,,
66.0,60,1,1,60,60
122.0,50,1,1,50,50
88.0,50,1,1,50,50
97.0,50,1,1,50,50
99.0,50,1,1,50,50
91.0,50,1,1,50,50
96.0,50,1,1,50,50
69.0,50,1,1,50,50
98.0,48,1,1,48,48


id_tom esteja identificando algum produto ou conjunto de tons, e não o tom individual.

ex: O id_tom = 66 está associado ao produto "10 Hour Wear Perfection Foundation", da Sephora Collection, e esse produto possui 60 registros de tons.
Isso é completamente diferente de dizer que id_tom identifica cada tom.

id_tom provavelmente é um identificador interno do produto/conjunto de tons, apesar do nome sugerir que seria um identificador do tom.

In [25]:
# ==============================================================================
# INSPEÇÃO DOS REGISTROS ASSOCIADOS A UM id_tom
# ==============================================================================

# Escolhemos um dos IDs que aparece muitas vezes para entender
# quais registros estão compartilhando o mesmo id_tom.

id_exemplo = 66

df_id[df_id["id_tom_marca"] == id_exemplo][
    [
        "id_tom_marca",
        "marca",
        "produto",
        "nome_tom",
        "tom_especifico",
        "luminosidade",
        "codigo_hex",
        "numero_tom"
    ]
].sort_values("numero_tom")

,id_tom_marca,marca,produto,nome_tom,tom_especifico,luminosidade,codigo_hex,numero_tom
263,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Pearl,03,0.864706,#EAD5CF,3.0
374,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Porcelain,04,0.841176,#EACDC3,4.0
489,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Porcelain,05,0.739216,#DEB39B,5.0
705,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Ivory,8,0.731373,#D9B19C,8.0
838,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Ivory,10,0.684314,#DCAA81,10.0
931,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Pink Ecru,11.5,0.786275,#DBBFB6,11.5
959,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Ecru,12,0.670588,#D6AC80,12.0
994,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Pink Porcelaine,13,0.698039,#D3AA91,13.0
1030,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Light Delicate Beige,14,0.709804,#DDAF8D,14.0
1060,66.0,SEPHORA COLLECTION,10 Hour Wear Perfection Foundation,Peach Cream,15,0.688235,#CDA792,15.0


Portanto, dentro daquele produto, cada registro representa um tom diferente.
id_tom > marca + produto > vários tons
Logo id_tom parece identificar o produto/conjunto de tons.

In [26]:
# Criamos uma chave temporária somente para investigação.
df_id["chave_negocio"] = (
    df_id["marca"].astype(str) + " | " +
    df_id["produto"].astype(str) + " | " +
    df_id["tom_especifico"].astype(str)
)

# Verificamos quantas chaves de negócio diferentes existem
# dentro de cada id_tom.
relacao_id_tom = (
    df_id
    .groupby("id_tom_marca")
    .agg(
        qtd_registros=("id_tom_marca", "size"),
        qtd_chaves_negocio=("chave_negocio", "nunique")
    )
    .sort_values("qtd_registros", ascending=False)
)

relacao_id_tom.head(20)

,qtd_registros,qtd_chaves_negocio
id_tom_marca,,
66.0,60,60
122.0,50,50
88.0,50,50
97.0,50,50
99.0,50,50
91.0,50,50
96.0,50,50
69.0,50,50
98.0,48,48


In [31]:
df_id = df_id.drop(columns="chave_negocio")

In [27]:
# ==============================================================================
# VALIDAÇÃO DA RELAÇÃO ENTRE id_tom E PRODUTO
# ==============================================================================

# Objetivo:
# Verificar se cada combinação de marca + produto está associada
# a apenas um único id_tom.
#
# Se o resultado for 1 para todas as combinações, teremos uma forte
# evidência de que o id_tom funciona como identificador do produto.
# ==============================================================================

produto_id_tom = (
    df_id
    .groupby(["marca", "produto"])["id_tom_marca"]
    .nunique()
    .sort_values(ascending=False)
)

print("Maior quantidade de id_tom por produto:",
      produto_id_tom.max())

print("\nProdutos associados a mais de um id_tom:")
print(produto_id_tom[produto_id_tom > 1])

Maior quantidade de id_tom por produto: 1

Produtos associados a mais de um id_tom:
Series([], Name: id_tom_marca, dtype: int64)


Temos agora uma evidencia mt forte de que: marca + produto <> id_tom é uma relação 1:1.

In [28]:
# ==============================================================================
# VALIDAÇÃO INVERSA: id_tom → MARCA + PRODUTO
# ==============================================================================

id_tom_produto = (
    df_id
    .groupby("id_tom_marca")
    .agg(
        marcas=("marca", "nunique"),
        produtos=("produto", "nunique")
    )
)

print("Maior quantidade de marcas por id_tom:",
      id_tom_produto["marcas"].max())

print("Maior quantidade de produtos por id_tom:",
      id_tom_produto["produtos"].max())

print("\nIDs associados a mais de um produto:")
print(
    id_tom_produto[
        (id_tom_produto["marcas"] > 1) |
        (id_tom_produto["produtos"] > 1)
    ]
)

Maior quantidade de marcas por id_tom: 1
Maior quantidade de produtos por id_tom: 1

IDs associados a mais de um produto:
Empty DataFrame
Columns: [marcas, produtos]
Index: []


Cada id_tom preenchido está associado a exatamente uma marca e um produto.

Isso é uma evidência muito forte de que o id_tom está funcionando como um identificador do produto/conjunto de tons, e não como identificador individual de cada tom.

Portanto:

id_tom
Identifica o produto.

tom_especifico
Identifica o tom dentro daquele produto.

numero_tom
Representa a numeração daquele tom.

In [33]:
# ==============================================================================
# VALIDAÇÃO DA RELAÇÃO: MARCA + PRODUTO → id_tom
# ==============================================================================

# Objetivo:
# Verificar se cada produto possui apenas um único id_tom.
#
# Se o resultado máximo for 1 e não existirem produtos associados
# a mais de um id_tom, teremos uma relação 1:1 entre:
#
#       marca + produto  ↔  id_tom
#
# Isso reforçaria que o id_tom funciona como identificador do produto.
# ==============================================================================

produto_id_tom = (
    df_id
    .groupby(["marca", "produto"])["id_tom_marca"]
    .nunique()
    .sort_values(ascending=False)
)

print("Maior quantidade de id_tom por produto:",
      produto_id_tom.max())

print("\nProdutos associados a mais de um id_tom:")

print(
    produto_id_tom[
        produto_id_tom > 1
    ]
)

Maior quantidade de id_tom por produto: 1

Produtos associados a mais de um id_tom:
Series([], Name: id_tom_marca, dtype: int64)


Existe uma relação 1:1 entre marca + produto e id_tom, considerando os registros em que id_tom está preenchido.

## Renomeação de `id_tom` para `id_produto`

### Justificativa

Durante a análise exploratória, verificou-se que o campo `id_tom`:

- possui valores duplicados entre os registros;
- está associado a uma única marca;
- está associado a um único produto;
- pode estar associado a diversos tons do mesmo produto.

Também foi validado que cada combinação de `marca + produto`
está associada a no máximo um `id_tom`.

Dessa forma, o comportamento observado indica que o campo atua
como identificador do produto/conjunto de tons, e não como
identificador individual de um tom.

Para evitar ambiguidade na etapa de modelagem do banco de dados,
a coluna será renomeada para `id_produto`.

In [44]:
# ==============================================================================
# RENOMEAÇÃO DE COLUNA
# ==============================================================================

# O campo original "id_tom" apresentou comportamento de identificador
# do produto, e não de identificador individual do tom.
#
# A renomeação não altera os valores da coluna.
# Apenas torna seu significado mais claro para as próximas etapas do projeto.
# ==============================================================================

df_numbers_limpo = df_numbers_limpo.rename(
    columns={
        "id_tom_marca":"id_produto_marca"
    }
)

In [46]:
display(df_numbers_limpo.columns.tolist())

['marca',
 'produto',
 'nome_tom',
 'tom_especifico',
 'luminosidade',
 'codigo_hex',
 'ordem_luminosidade',
 'numero_tom',
 'id_produto_marca',
 'tom_unificado']

In [45]:
# ==============================================================================
# ANÁLISE DOS NULOS EM id_produto
# ==============================================================================

# Objetivo:
# Identificar quais marcas e produtos possuem registros sem id_produto.
#
# Antes de decidir o tratamento dos nulos, precisamos entender
# se eles estão concentrados em determinados produtos ou marcas.
# ==============================================================================

nulos_id_produto = df_numbers_limpo[
    df_numbers_limpo["id_produto_marca"].isna()
]

print("Total de registros sem id_produto:",
      len(nulos_id_produto))

print("\nQuantidade de marcas afetadas:",
      nulos_id_produto["marca"].nunique())

print("Quantidade de produtos afetados:",
      nulos_id_produto["produto"].nunique())

Total de registros sem id_produto: 126

Quantidade de marcas afetadas: 8
Quantidade de produtos afetados: 15


In [346]:
# Quantidade de registros sem id_produto por marca
nulos_por_marca = (
    nulos_id_produto
    .groupby("marca")
    .size()
    .sort_values(ascending=False)
)

print(nulos_por_marca)

marca
Clinique              119
Benefit Cosmetics       1
Black Up                1
Erborian                1
Lancôme                 1
Marc Jacobs Beauty      1
Shiseido                1
bareMinerals            1
dtype: int64


Isso é um forte indício de problema ou ausência sistemática na origem, principalmente no caso da Clinique.

In [347]:
# Quantidade de registros sem id_produto por produto
nulos_por_produto = (
    nulos_id_produto
    .groupby(["marca", "produto"])
    .size()
    .sort_values(ascending=False)
)

print(nulos_por_produto)

marca               produto                                                          
Clinique            Beyond Perfecting Foundation + Concealer                             36
                    Even Better Glow Light Reflecting Makeup Broad Spectrum SPF 15       26
                    Stay Matte Oil-Free Makeup                                           20
                    Beyond Perfecting Powder Foundation + Concealer                      12
                    Superbalanced Makeup                                                 11
                    Even Better Refresh™ Hydrating and Repairing Foundation               7
                    Almost Powder Makeup                                                  6
Black Up            Full Coverage Cream Foundation                                        1
Benefit Cosmetics   Hello Happy Soft Blur Foundation                                      1
Clinique            Age Defense BB Cream Broad Spectrum SPF 30                        

In [348]:
# ==============================================================================
# VERIFICAÇÃO DE RECUPERAÇÃO DO id_produto
# ==============================================================================

# Criamos uma tabela que mostra, para cada produto:
# - quantos registros existem;
# - quantos id_produto estão preenchidos;
# - quantos estão nulos;
# - quantos id_produto diferentes existem.
# ==============================================================================

analise_produto_id = (
    df_numbers_limpo
    .groupby(["marca", "produto"])
    .agg(
        total_registros=("produto", "size"),
        id_produto_preenchidos=("id_produto", "count"),
        id_produto_unicos=("id_produto", "nunique")
    )
)

analise_produto_id["id_produto_nulos"] = (
    analise_produto_id["total_registros"]
    - analise_produto_id["id_produto_preenchidos"]
)

analise_produto_id

total_registros  \
marca                   produto                                                               
AMOREPACIFIC            Color Control Cushion Compact Broad Spectrum SP...                5   
Almay                   Clear Complexion Make Myself Clear Makeup                         8   
                        Skin Perfecting Healthy Biome Makeup                              6   
                        Smart Shade Skintone Matching Makeup                              3   
Anastasia Beverly Hills Luminous Foundation                                              50   
...                                                                                     ...   
bareMinerals            bareSkin® Pure Brightening Serum Foundation Bro...                1   
beautyblender           Bounce™ Liquid Whip Long Wear Foundation                         40   
florence by mills       Like a Light Skin Tint                                           15   
jane iredale            Beyond Matte Liquid Foundation                                   18   
surratt beauty          Surreal Skin Foundation Wand                                     14   

                                                                            id_produto_preenchidos  \
marca                   produto                                                                      
AMOREPACIFIC            Color Control Cushion Compact Broad Spectrum SP...                       5   
Almay                   Clear Complexion Make Myself Clear Makeup                                8   
                        Skin Perfecting Healthy Biome Makeup                                     6   
                        Smart Shade Skintone Matching Makeup                                     3   
Anastasia Beverly Hills Luminous Foundation                                                     50   
...                                                                                            ...   
bareMinerals            bareSkin® Pure Brightening Serum Foundation Bro...                       0   
beautyblender           Bounce™ Liquid Whip Long Wear Foundation                                40   
florence by mills       Like a Light Skin Tint                                                  15   
jane iredale            Beyond Matte Liquid Foundation                                          18   
surratt beauty          Surreal Skin Foundation Wand                                            14   

                                                                            id_produto_unicos  \
marca                   produto                                                                 
AMOREPACIFIC            Color Control Cushion Compact Broad Spectrum SP...                  1   
Almay                   Clear Complexion Make Myself Clear Makeup                           1   
                        Skin Perfecting Healthy Biome Makeup                                1   
                        Smart Shade Skintone Matching Makeup                                1   
Anastasia Beverly Hills Luminous Foundation                                                 1   
...                                                                                       ...   
bareMinerals            bareSkin® Pure Brightening Serum Foundation Bro...                  0   
beautyblender           Bounce™ Liquid Whip Long Wear Foundation                            1   
florence by mills       Like a Light Skin Tint                                              1   
jane iredale            Beyond Matte Liquid Foundation                                      1   
surratt beauty          Surreal Skin Foundation Wand                                        1   

                                                                            id_produto_nulos  
marca                   produto                                                               
AMOREPACIFIC            Color Control Cushion Compact Broad S

In [349]:
# Produtos que possuem tanto registros preenchidos
# quanto registros nulos de id_produto.

recuperaveis = analise_produto_id[
    (analise_produto_id["id_produto_nulos"] > 0) &
    (analise_produto_id["id_produto_preenchidos"] > 0)
]

print(recuperaveis)

Empty DataFrame
Columns: [total_registros, id_produto_preenchidos, id_produto_unicos, id_produto_nulos]
Index: []


não existe um produto que tenha simultaneamente IDs preenchidos e nulos.
Significa que não encontramos nenhum produto parcialmente preenchido.

In [350]:
# ==============================================================================
# VERIFICAÇÃO DE DUPLICIDADE DE PRODUTO ENTRE REGISTROS COM E SEM id_produto
# ==============================================================================

# Objetivo:
# Verificar se algum produto aparece tanto com id_produto preenchido
# quanto sem id_produto.
#
# Essa análise complementa o teste anterior e ajuda a confirmar
# se os nulos representam produtos sem identificador na origem.
# ==============================================================================

produtos_com_id = (
    df_numbers_limpo[df_numbers_limpo["id_produto"].notna()]
    .set_index(["marca", "produto"])
)

produtos_sem_id = (
    df_numbers_limpo[df_numbers_limpo["id_produto"].isna()]
    .set_index(["marca", "produto"])
)

produtos_em_ambos = produtos_sem_id.index.intersection(
    produtos_com_id.index
)

print(
    "Produtos encontrados tanto com quanto sem id_produto:",
    len(produtos_em_ambos)
)

print("\nProdutos encontrados nos dois grupos:")
print(produtos_em_ambos)

Produtos encontrados tanto com quanto sem id_produto: 0

Produtos encontrados nos dois grupos:
MultiIndex([], names=['marca', 'produto'])


os 126 nulos pertencem a produtos que não possuem id_produto na própria base.

In [351]:
# ==============================================================================
# VALIDAÇÃO DA CHAVE NATURAL DO PRODUTO
# ==============================================================================

# Verifica se cada combinação de marca + produto representa
# um único produto lógico na base.
#
# Essa combinação será utilizada para identificar os produtos
# que precisam receber um ID interno.

produto_natural = (
    df_numbers_limpo
    .groupby(["marca", "produto"])
    .size()
    .reset_index(name="qtd_registros")
)

print("Total de produtos distintos:",
      len(produto_natural))

print("\nTotal de combinações marca + produto:",
      df_numbers_limpo[["marca", "produto"]].drop_duplicates().shape[0])

Total de produtos distintos: 145

Total de combinações marca + produto: 145


marca + produto identifica unicamente cada produto da base.
Então podemos usar essa combinação como nossa chave natural para gerar um id_produto interno.

In [352]:
# ==============================================================================
# PRESERVAÇÃO DO IDENTIFICADOR ORIGINAL
# ==============================================================================

# O campo "id_produto" atualmente contém o identificador
# que veio originalmente da fonte de dados.
#
# Como iremos criar um novo identificador interno para o nosso modelo,
# precisamos preservar o ID original em uma coluna separada.
# ==============================================================================

df_numbers_limpo = df_numbers_limpo.rename(
    columns={
        "id_produto": "id_produto_origem"
    }
)

print(df_numbers_limpo.columns.tolist())

['marca', 'produto', 'nome_tom', 'tom_especifico', 'luminosidade', 'codigo_hex', 'ordem_luminosidade', 'numero_tom', 'id_produto_origem', 'tom_unificado']


In [353]:
# ==============================================================================
# CRIAÇÃO DA DIMENSÃO DE PRODUTOS
# ==============================================================================

# Como validamos que "marca + produto" identifica unicamente um produto,
# podemos criar uma tabela contendo apenas os produtos distintos.
#
# Cada combinação de marca + produto receberá um id_produto interno.
# ==============================================================================

df_produtos = (
    df_numbers_limpo[
        ["marca", "produto", "id_produto_origem"]
    ]
    .drop_duplicates(subset=["marca", "produto"])
    .reset_index(drop=True)
)

print("Total de produtos:", len(df_produtos))

Total de produtos: 145


In [354]:
# ==============================================================================
# CRIAÇÃO DO IDENTIFICADOR INTERNO DO PRODUTO
# ==============================================================================

# O índice começa em 0 no Pandas.
# Por isso, adicionamos 1 para que o primeiro ID seja 1.
#
# Esse identificador NÃO veio da fonte.
# Ele foi criado pelo nosso ETL para servir como chave interna.
# ==============================================================================

df_produtos["id_produto"] = (
    df_produtos.index + 1
)

In [355]:
# ==============================================================================
# ORDENAÇÃO DETERMINÍSTICA DOS PRODUTOS
# ==============================================================================

# Ordenamos os produtos por marca e produto antes de gerar o ID.
#
# Dessa forma, a atribuição dos IDs não dependerá da ordem original
# dos registros no CSV.
# ==============================================================================

df_produtos = (
    df_produtos
    .sort_values(
        ["marca", "produto"]
    )
    .reset_index(drop=True)
)

df_produtos["id_produto"] = (
    df_produtos.index + 1
)

In [356]:
# ==============================================================================
# VALIDAÇÃO DA DIMENSÃO PRODUTO
# ==============================================================================

print("Total de produtos:", len(df_produtos))

print(
    "IDs internos únicos:",
    df_produtos["id_produto"].nunique()
)

print(
    "IDs de origem preenchidos:",
    df_produtos["id_produto_origem"].notna().sum()
)

print(
    "IDs de origem nulos:",
    df_produtos["id_produto_origem"].isna().sum()
)

Total de produtos: 145
IDs internos únicos: 145
IDs de origem preenchidos: 130
IDs de origem nulos: 15


In [357]:
# ==============================================================================
# ASSOCIAÇÃO DO id_produto INTERNO AOS REGISTROS DE TONS
# ==============================================================================

# Utilizamos "marca + produto" como chave natural para localizar
# o id_produto interno correspondente a cada registro.
# ==============================================================================

df_numbers_limpo = df_numbers_limpo.drop(
    columns=["id_produto_origem"]
).merge(
    df_produtos[
        ["id_produto", "id_produto_origem", "marca", "produto"]
    ],
    on=["marca", "produto"],
    how="left"
)

In [358]:
# ==============================================================================
# VALIDAÇÃO DO id_produto INTERNO
# ==============================================================================

print(
    "Total de registros:",
    len(df_numbers_limpo)
)

print(
    "Registros sem id_produto interno:",
    df_numbers_limpo["id_produto"].isna().sum()
)

print(
    "IDs de produtos internos:",
    df_numbers_limpo["id_produto"].nunique()
)

Total de registros: 3117
Registros sem id_produto interno: 0
IDs de produtos internos: 145


id_produto
É nosso identificador confiável.

id_produto_origem
É o identificador fornecido pela fonte, quando disponível.

# Construção do Identificador Interno do Produto

## Objetivo

Criar um identificador interno (`id_produto`) para os 145 produtos
da base, independentemente da existência de um identificador na
fonte de origem.

O identificador original foi preservado em `id_produto_origem`.

## Chave natural

A análise exploratória confirmou que:

`marca + produto`

identifica unicamente os 145 produtos da base.

## Resultado

- 145 produtos distintos;
- 145 `id_produto` internos;
- 130 produtos possuem `id_produto_origem`;
- 15 produtos não possuem identificador na origem;
- nenhum registro permanece sem `id_produto` interno;
- os 126 registros que não possuíam ID de origem foram preservados.

In [359]:
# =========================================================
# REMOÇÃO DE COLUNAS IRRELEVANTES PARA A ANÁLISE
# =========================================================

# Remove as colunas auxiliares que não serão utilizadas nos cálculos/gráficos
df_numbers_limpo = df_numbers_limpo.drop(columns=['ordem_luminosidade'], errors='ignore')

# Exibe as colunas restantes na base tratada
print("Colunas mantidas no DataFrame de análise:")
print(df_numbers_limpo.columns.tolist())

Colunas mantidas no DataFrame de análise:
['marca', 'produto', 'nome_tom', 'tom_especifico', 'luminosidade', 'codigo_hex', 'numero_tom', 'tom_unificado', 'id_produto', 'id_produto_origem']


### Decisão de Limpeza: Descarte de Variáveis Auxiliares (`ordem_luminosidade`)

Após diagnóstico detalhado das variáveis complementares da base `allNumbers`:

1. **`ordem_luminosidade` (`lightToDark`):** Trata-se de uma variável booleana que indica se a numeração da marca segue a ordem do tom claro ao escuro. Por não impactar as métricas de variabilidade de luminosidade, agrupamentos por categoria (`Grupo_de_Tom`) ou decisões de portfólio, foi opcionalmente descartada da modelagem analítica.

In [360]:
# Verificando se temos dados duplicados
df_numbers_limpo.duplicated().sum()

np.int64(0)

In [361]:
df_numbers_limpo.head()

,marca,produto,nome_tom,tom_especifico,luminosidade,codigo_hex,numero_tom,tom_unificado,id_produto,id_produto_origem
0,Makeup Revolution,Conceal & Define Full Coverage Foundation,NaN,F0,0.949020,#F2F2F2,0.0,F0,83,1.0
1,HOURGLASS,Veil Fluid Makeup,Porcelain,No. 0,0.817647,#F6D3AB,0.0,No. 0,50,2.0
2,TOM FORD,Traceless Soft Matte Foundation,Pearl,0.0,0.850980,#F0D8C2,0.0,0.0,123,3.0
3,Armani Beauty,Neo Nude Foundation,NaN,0,0.911765,#F0E8E1,0.0,0,10,4.0
4,TOM FORD,Traceless Foundation Stick,Pearl,0.0,0.911765,#FDE5D4,0.0,0.0,122,5.0


In [362]:
# Extraindo informações estatísticas da coluna 'luminosidade' do DataFrame 'df_numbers', como média, desvio padrão, valores mínimo e máximo, entre outros.
print(df_numbers_limpo['luminosidade'].describe())

count    3117.000000
mean        0.634527
std         0.162132
min         0.154902
25%         0.519608
50%         0.666667
75%         0.756863
max         0.994118
Name: luminosidade, dtype: float64


In [363]:
# ------------------------------------------------------------------------------
# ADEQUAÇÃO DA ESCALA DE LUMINOSIDADE (DE DECIMAL PARA PERCENTUAL 0-100)
# ------------------------------------------------------------------------------
# A variável 'luminosidade' varia de 0.0 a 1.0. Multiplicamos por 100 para 
# atender à regra de negócio (faixas de 0 a 100) especificada no case.
df_numbers_limpo['luminosidade'] = df_numbers_limpo['luminosidade'] * 100

In [364]:
# ------------------------------------------------------------------------------
# 3. CRIAÇÃO DAS FAIXAS (BINS) E RÓTULOS (MISSÃO 1 DO CASE)
# ------------------------------------------------------------------------------
# Regra do Desafio:
#  0 a 30  -> Retinto / Deep
# 31 a 50  -> Escuro / Dark
# 51 a 75  -> Médio / Medium
# 76 a 100 -> Claro / Light

# 1. BINS (PONTOS DE CORTE / FAiXAS NUMÉRICAS):
# Definimos os limites numéricos para fatiar a coluna 'luminosidade'.
# Nota: Usamos -1 ao invés de 0 no início para garantir que o valor '0' 
# seja incluído na primeira faixa (de 0 a 30).
# As 4 faixas criadas são: (-1 a 30], (30 a 50], (50 a 75], (75 a 100]
bins = [-1, 30, 50, 75, 100]
# 2. LABELS (ETIQUETAS DE TEXTO / RÓTULOS):
# Definimos os nomes que cada uma das 4 faixas numéricas vai receber.
# A ordem aqui DEVE corresponder perfeitamente à ordem das faixas dos bins.
labels = ['Retinto / Deep', 'Escuro / Dark', 'Médio / Medium', 'Claro / Light']

# 3. AGRUPAMENTO COM PD.CUT (CORTANDO OS DADOS E CRIANDO A NOVA COLUNA):
# O pd.cut pega a coluna 'luminosidade', compara cada número com as faixas em 'bins',
# e substitui pelo nome correspondente presente em 'labels'.
# O resultado é guardado na NOVA coluna chamada 'grupo_de_tom'.
df_numbers_limpo['grupo_de_tom'] = pd.cut(
    df_numbers_limpo['luminosidade'], # Coluna original com os números (0 a 100)
    bins=bins, # Limites numéricos onde o corte é feito
    labels=labels # Nomes em texto que serão atribuídos a cada faixa
)

# Visualização do resultado final: Exibe as primeiras 10 linhas das colunas 'luminosidade' e 'grupo_de_tom' para verificar se a categorização foi aplicada corretamente.
print(df_numbers_limpo[['luminosidade', 'grupo_de_tom']].head(15))

    luminosidade    grupo_de_tom
0      94.901961   Claro / Light
1      81.764706   Claro / Light
2      85.098039   Claro / Light
3      91.176471   Claro / Light
4      91.176471   Claro / Light
5      73.137255  Médio / Medium
6      82.156863   Claro / Light
7      83.137255   Claro / Light
8      81.372549   Claro / Light
9      90.980392   Claro / Light
10     91.960784   Claro / Light
11     75.686275   Claro / Light
12     84.705882   Claro / Light
13     88.431373   Claro / Light
14     80.000000   Claro / Light


In [365]:
# 4. VALIDAÇÃO DO RESULTADO DA CATEGORIZAÇÃO
print("\n--- Distribuição das Bases por Categoria de Tom (Visão do Mercado) ---")
print(df_numbers_limpo['grupo_de_tom'].value_counts(dropna=False))

print("\n--- Percentual do Mercado por Categoria ---")
print((df_numbers_limpo['grupo_de_tom'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')


--- Distribuição das Bases por Categoria de Tom (Visão do Mercado) ---
grupo_de_tom
Médio / Medium    1592
Claro / Light      817
Escuro / Dark      612
Retinto / Deep      96
Name: count, dtype: int64

--- Percentual do Mercado por Categoria ---
grupo_de_tom
Médio / Medium    51.07%
Claro / Light     26.21%
Escuro / Dark     19.63%
Retinto / Deep     3.08%
Name: proportion, dtype: str


In [366]:
# Retorna a quantidade exata de linhas com valor ausente (NaN)
total_nulos = df_numbers_limpo['grupo_de_tom'].isna().sum()
print(f"Total de valores ausentes: {total_nulos}")

Total de valores ausentes: 0


PADRONIZAÇÃO DOS CONTEÚDOS PRA MELHOR UTILIZAÇÃO NO SQL

In [367]:
# 6. Padroniza o conteúdo das células de texto em minúsculo (exceto 'grupo_de_tom')
colunas_conteudo_texto = ['marca', 'produto', 'nome_tom', 'tom_especifico', 'codigo_hex', 'tom_unificado']
for col in colunas_conteudo_texto:
    # 1. .str.strip() -> Remove espaços no início e no final do texto (ex: " tom " vira "tom")
    # 2. .str.lower() -> Transforma todo o texto em minúsculo (ex: "Ruby" vira "ruby")
    df_numbers_limpo[col] = df_numbers_limpo[col].str.strip().str.lower()

In [368]:
# Cria uma coluna de ID numérico sequencial (1, 2, 3...)
df_numbers_limpo['id_numbers'] = range(1, len(df_numbers_limpo) + 1)

# Reorganiza as colunas para colocar o 'id_numbers' como a primeira coluna da tabela
colunas = ['id_numbers'] + [col for col in df_numbers_limpo.columns if col != 'id_numbers']
df_numbers_limpo = df_numbers_limpo[colunas]

In [369]:
df_numbers_limpo.head()

,id_numbers,marca,produto,nome_tom,tom_especifico,luminosidade,codigo_hex,numero_tom,tom_unificado,id_produto,id_produto_origem,grupo_de_tom
0,1,makeup revolution,conceal & define full coverage foundation,NaN,f0,94.901961,#f2f2f2,0.0,f0,83,1.0,Claro / Light
1,2,hourglass,veil fluid makeup,porcelain,no. 0,81.764706,#f6d3ab,0.0,no. 0,50,2.0,Claro / Light
2,3,tom ford,traceless soft matte foundation,pearl,0.0,85.098039,#f0d8c2,0.0,0.0,123,3.0,Claro / Light
3,4,armani beauty,neo nude foundation,NaN,0,91.176471,#f0e8e1,0.0,0,10,4.0,Claro / Light
4,5,tom ford,traceless foundation stick,pearl,0.0,91.176471,#fde5d4,0.0,0.0,122,5.0,Claro / Light


In [47]:
# Salva o CSV na pasta processed
df_numbers_limpo.to_csv('../data/processed/df_numbers_limpo.csv', index=False)